# Example 1: Reference Tracking

This notebook uses the same double-integrator dynamics as the lecture example:

$$x(k+1)=Ax(k)+Bu(k),$$

$$A=\begin{bmatrix}1 & 1\\0 & 1\end{bmatrix},\qquad B=\begin{bmatrix}0\\1\end{bmatrix},\qquad H=I.$$

The comparison shows:
- left: nominal MPC regulating to the origin,
- right: reference-tracking MPC optimizing deviations around a steady-state target.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
import cvxpy as cp
from IPython.display import display

try:
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    pass

plt.rcParams["figure.figsize"] = (14, 10)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 11

A = np.array([[1.0, 1.0], [0.0, 1.0]])
B = np.array([[0.0], [1.0]])
H = np.eye(2)

x_min = np.array([-5.0, -5.0])
x_max = np.array([5.0, 5.0])
u_min = -0.5
u_max = 0.5

Q_tracking = np.eye(2)
R_tracking = np.array([[10.0]])
P_tracking = Q_tracking.copy()

x0_tracking = np.array([2.0, 0.0])

def solve_steady_state_target(r):
    x_s = np.array(r, dtype=float)
    if np.any(x_s < x_min - 1e-8) or np.any(x_s > x_max + 1e-8):
        return None, None
    u_s = 0.0
    if u_s < u_min - 1e-8 or u_s > u_max + 1e-8:
        return None, None
    if not np.allclose(x_s, A @ x_s + B[:, 0] * u_s, atol=1e-8):
        return None, None
    return x_s, u_s

def solve_mpc_step_tracking(x0, horizon, x_s=None, u_s=None, solver=cp.OSQP):
    X = cp.Variable((2, horizon + 1))
    U = cp.Variable((1, horizon))
    constraints = [X[:, 0] == x0]
    cost = 0
    for k in range(horizon):
        if x_s is None:
            cost += cp.quad_form(X[:, k], Q_tracking) + cp.quad_form(U[:, k], R_tracking)
        else:
            cost += cp.quad_form(X[:, k] - x_s, Q_tracking) + cp.quad_form(U[:, k] - u_s, R_tracking)
        constraints += [
            X[:, k + 1] == A @ X[:, k] + B @ U[:, k],
            X[:, k] >= x_min,
            X[:, k] <= x_max,
            U[:, k] >= u_min,
            U[:, k] <= u_max,
        ]
    if x_s is None:
        cost += cp.quad_form(X[:, horizon], P_tracking)
    else:
        cost += cp.quad_form(X[:, horizon] - x_s, P_tracking)
    problem = cp.Problem(cp.Minimize(cost), constraints)
    problem.solve(solver=solver, warm_start=True, verbose=False, eps_abs=1e-5, eps_rel=1e-5, max_iter=20000)
    if U.value is None:
        return None, problem.status
    return float(U.value[0, 0]), problem.status

def simulate_mpc_tracking(x0, horizon, sim_steps, x_s=None, u_s=None):
    x = np.array(x0, dtype=float).copy()
    xs = [x.copy()]
    us = []
    statuses = []
    for _ in range(sim_steps):
        u_cmd, status = solve_mpc_step_tracking(x, horizon, x_s=x_s, u_s=u_s)
        statuses.append(status)
        if u_cmd is None:
            break
        x = A @ x + B[:, 0] * u_cmd
        xs.append(x.copy())
        us.append(u_cmd)
    return {
        "x": np.array(xs),
        "u": np.array(us),
        "statuses": statuses,
    }

def classify_initial_point(x0, horizon, sim_steps, x_s=None, u_s=None):
    x = np.array(x0, dtype=float).copy()
    xs = [x.copy()]
    us = []
    statuses = []
    for k in range(sim_steps):
        u_cmd, status = solve_mpc_step_tracking(x, horizon, x_s=x_s, u_s=u_s)
        statuses.append(status)
        if u_cmd is None:
            return (0 if k == 0 else 1, {"x": np.array(xs), "u": np.array(us), "statuses": statuses})
        x = A @ x + B[:, 0] * u_cmd
        xs.append(x.copy())
        us.append(u_cmd)
    return (2, {"x": np.array(xs), "u": np.array(us), "statuses": statuses})

def evaluate_convergence_grid(ref_x1, horizon, sim_steps, x_s=None, u_s=None, grid_n=15, progress=None):
    xs = np.linspace(-4.0, 4.0, grid_n)
    ys = np.linspace(-4.0, 4.0, grid_n)
    X1, X2 = np.meshgrid(xs, ys)
    class_map = np.zeros_like(X1, dtype=int)
    total = grid_n * grid_n
    if progress is not None:
        progress.max = total
        progress.value = 0
    for i in range(grid_n):
        for j in range(grid_n):
            x0 = np.array([xs[i], ys[j]])
            cls, _ = classify_initial_point(x0, horizon, sim_steps, x_s=x_s, u_s=u_s)
            class_map[j, i] = cls
            if progress is not None:
                progress.value += 1
    return X1, X2, class_map


In [ ]:
def classify_point(x0, horizon, sim_steps, x_s=None, u_s=None, tolerance=0.25):
    x = np.array(x0, dtype=float).copy()
    xs = [x.copy()]
    us = []
    statuses = []
    for k in range(sim_steps):
        u_cmd, status = solve_mpc_step_tracking(x, horizon, x_s=x_s, u_s=u_s)
        statuses.append(status)
        if u_cmd is None:
            return 0, {"x": np.array(xs), "u": np.array(us), "statuses": statuses}
        x = A @ x + B[:, 0] * u_cmd
        xs.append(x.copy())
        us.append(u_cmd)

    target = x_s if x_s is not None else np.zeros(2)
    final_error = np.linalg.norm(xs[-1] - target)
    successful = final_error <= tolerance
    return (2 if successful else 1), {
        "x": np.array(xs),
        "u": np.array(us),
        "statuses": statuses,
        "final_error": final_error,
        "target": target,
    }


def evaluate_convergence_grid(horizon, sim_steps, x_s=None, u_s=None, grid_n=15, progress=None):
    xs = np.linspace(-4.0, 4.0, grid_n)
    ys = np.linspace(-4.0, 4.0, grid_n)
    X1, X2 = np.meshgrid(xs, ys)
    class_map = np.zeros_like(X1, dtype=int)
    total = grid_n * grid_n
    if progress is not None:
        progress.max = total
        progress.value = 0
    for i in range(grid_n):
        for j in range(grid_n):
            x0 = np.array([xs[i], ys[j]])
            cls, _ = classify_point(x0, horizon, sim_steps, x_s=x_s, u_s=u_s)
            class_map[j, i] = cls
            if progress is not None:
                progress.value += 1
    return X1, X2, class_map


def generate_sample_trajectories(horizon, sim_steps, x_s=None, u_s=None, sample_n=9, progress=None):
    xs = np.linspace(-4.0, 4.0, sample_n)
    ys = np.linspace(-4.0, 4.0, sample_n)
    trajectories = []
    total = sample_n * sample_n
    count = 0
    if progress is not None:
        progress.max = total
        progress.value = 0
    for x1 in xs:
        for x2 in ys:
            x0 = np.array([x1, x2], dtype=float)
            cls, sim = classify_point(x0, horizon, sim_steps, x_s=x_s, u_s=u_s)
            trajectories.append({"class": cls, "sim": sim, "x0": x0})
            count += 1
            if progress is not None:
                progress.value = count
    return trajectories


def plot_convergence_map(ax, X1, X2, class_map, trajectories, target_point, title):
    cmap_colors = np.array([
        [0.86, 0.86, 0.86, 1.0],
        [0.98, 0.76, 0.53, 1.0],
        [0.66, 0.86, 0.66, 1.0],
    ])
    ax.contourf(X1, X2, class_map, levels=[-0.5, 0.5, 1.5, 2.5], colors=cmap_colors)
    ax.contour(X1, X2, (class_map > 0).astype(float), levels=[0.5], colors="black", linewidths=1.8)

    for item in trajectories:
        xs = item["sim"]["x"]
        cls = item["class"]
        if len(xs) <= 1:
            ax.scatter(xs[0, 0], xs[0, 1], s=9, color="#444444", alpha=0.35)
            continue
        color = "#2e7d32" if cls == 2 else "#c96a00"
        ax.plot(xs[:, 0], xs[:, 1], color=color, lw=0.9, alpha=0.35)
        ax.scatter(xs[0, 0], xs[0, 1], s=10, color=color, alpha=0.35)

    ax.plot([target_point[0]], [target_point[1]], marker="x", color="black", ms=10, mew=2)
    ax.axvline(target_point[0], color="black", ls="--", lw=1.0, alpha=0.6)
    ax.axhline(target_point[1], color="black", ls="--", lw=1.0, alpha=0.6)

    ax.set_title(title)
    ax.set_xlabel("$x_1$")
    ax.set_xlim(-4.0, 4.0)
    ax.set_ylim(-4.0, 4.0)
    ax.grid(True, alpha=0.35)


def compare_tracking_scenarios(ref_x1, ref_x2, horizon=6, sim_steps=20):
    r = np.array([ref_x1, ref_x2])
    x_s, u_s = solve_steady_state_target(r)

    nominal_progress = widgets.IntProgress(value=0, min=0, max=1, description="Nominal map:")
    tracking_progress = widgets.IntProgress(value=0, min=0, max=1, description="Tracking map:")
    trajectory_progress = widgets.IntProgress(value=0, min=0, max=1, description="Trajectories:")
    progress_box = widgets.VBox([
        widgets.Label("Computing convergence maps..."),
        nominal_progress,
        tracking_progress,
        trajectory_progress,
    ])
    display(progress_box)

    grid_n = 15
    X1, X2, nominal_map = evaluate_convergence_grid(
        horizon,
        sim_steps,
        x_s=None,
        u_s=None,
        grid_n=grid_n,
        progress=nominal_progress,
    )
    _, _, tracking_map = evaluate_convergence_grid(
        horizon,
        sim_steps,
        x_s=x_s,
        u_s=u_s,
        grid_n=grid_n,
        progress=tracking_progress,
    )

    nominal_traj = generate_sample_trajectories(
        horizon,
        sim_steps,
        x_s=None,
        u_s=None,
        sample_n=9,
        progress=trajectory_progress,
    )
    tracking_traj = generate_sample_trajectories(
        horizon,
        sim_steps,
        x_s=x_s,
        u_s=u_s,
        sample_n=9,
        progress=trajectory_progress,
    )

    progress_box.children = [widgets.Label("Rendering result..."), nominal_progress, tracking_progress, trajectory_progress]

    fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)
    plot_convergence_map(
        axes[0],
        X1,
        X2,
        nominal_map,
        nominal_traj,
        np.zeros(2),
        "Nominal MPC (origin regulation)",
    )
    plot_convergence_map(
        axes[1],
        X1,
        X2,
        tracking_map,
        tracking_traj,
        r,
        "Reference-tracking MPC",
    )

    axes[0].set_ylabel("$x_2$")
    legend_handles = [
        plt.Line2D([0], [0], color="#bdbdbd", lw=8, label="Infeasible at $k=0$"),
        plt.Line2D([0], [0], color="#f99d44", lw=8, label="Feasible but not near target"),
        plt.Line2D([0], [0], color="#2e7d32", lw=8, label="Feasible and near target"),
        plt.Line2D([0], [0], color="black", lw=1.8, label="Initial feasible boundary"),
    ]
    fig.legend(handles=legend_handles, loc="lower center", ncol=4, framealpha=0.95)
    fig.tight_layout(rect=[0, 0.06, 1, 1])
    plt.show()

    summary = pd.DataFrame([
        {
            "scenario": "Nominal MPC",
            "infeasible k0": int(np.sum(nominal_map == 0)),
            "feasible then fails": int(np.sum(nominal_map == 1)),
            "feasible all steps": int(np.sum(nominal_map == 2)),
        },
        {
            "scenario": "Reference-tracking MPC",
            "infeasible k0": int(np.sum(tracking_map == 0)),
            "feasible then fails": int(np.sum(tracking_map == 1)),
            "feasible all steps": int(np.sum(tracking_map == 2)),
        },
    ])
    display(summary)


ref_x1_sl = widgets.FloatSlider(value=4.0, min=-4.0, max=4.0, step=0.5, description="reference x1", continuous_update=False)
ref_x2_sl = widgets.FloatSlider(value=0.0, min=-4.0, max=4.0, step=0.5, description="reference x2", continuous_update=False)
horizon_tracking_sl = widgets.IntSlider(value=4, min=2, max=12, step=1, description="Horizon", continuous_update=False)
sim_steps_tracking_sl = widgets.IntSlider(value=10, min=5, max=40, step=1, description="Sim steps", continuous_update=False)
tracking_out = widgets.Output()

def update_tracking_view(_=None):
    tracking_out.clear_output(wait=True)
    with tracking_out:
        compare_tracking_scenarios(ref_x1_sl.value, ref_x2_sl.value, horizon_tracking_sl.value, sim_steps_tracking_sl.value)

for widget_item in [ref_x1_sl, ref_x2_sl, horizon_tracking_sl, sim_steps_tracking_sl]:
    widget_item.observe(update_tracking_view, names="value")

controls = widgets.HBox([ref_x1_sl, ref_x2_sl, horizon_tracking_sl, sim_steps_tracking_sl])
display(widgets.VBox([controls, tracking_out]))
update_tracking_view()


## Example 2: Disturbance Rejection

This example keeps the same double-integrator dynamics as Example 1 and adds a constant disturbance through $B_d d$.

- Left: nominal MPC ignores the disturbance.
- Right: disturbance-aware MPC uses a known disturbance model during prediction.

The feasibility maps are fixed at horizon $N=4$ and $10$ closed-loop steps.
Only the initial state $(x_1, x_2)$ is adjustable.

In [ ]:
Bd = np.array([[0.0], [1.0]])
C = np.array([[1.0, 0.0]])
Cd = np.array([[0.0]])

d_true = 0.25
horizon_dist = 4
sim_steps_dist = 10
map_grid_n = 15
convergence_tolerance = 0.35


def solve_mpc_step_nominal(x0, horizon, solver=cp.OSQP):
    X = cp.Variable((2, horizon + 1))
    U = cp.Variable((1, horizon))
    constraints = [X[:, 0] == x0]
    cost = 0
    for k in range(horizon):
        cost += cp.quad_form(X[:, k], Q_tracking) + cp.quad_form(U[:, k], R_tracking)
        constraints += [
            X[:, k + 1] == A @ X[:, k] + B @ U[:, k],
            X[:, k] >= x_min,
            X[:, k] <= x_max,
            U[:, k] >= u_min,
            U[:, k] <= u_max,
        ]
    cost += cp.quad_form(X[:, horizon], P_tracking)
    problem = cp.Problem(cp.Minimize(cost), constraints)
    problem.solve(solver=solver, warm_start=True, verbose=False, eps_abs=1e-5, eps_rel=1e-5, max_iter=20000)
    if U.value is None:
        return None, problem.status, None
    return float(U.value[0, 0]), problem.status, X.value


def solve_mpc_step_disturbance(x0, d_hat, horizon, solver=cp.OSQP):
    X = cp.Variable((2, horizon + 1))
    U = cp.Variable((1, horizon))
    constraints = [X[:, 0] == x0]
    cost = 0
    for k in range(horizon):
        cost += cp.quad_form(X[:, k], Q_tracking) + cp.quad_form(U[:, k], R_tracking)
        constraints += [
            X[:, k + 1] == A @ X[:, k] + B @ U[:, k] + Bd[:, 0] * d_hat,
            X[:, k] >= x_min,
            X[:, k] <= x_max,
            U[:, k] >= u_min,
            U[:, k] <= u_max,
        ]
    cost += cp.quad_form(X[:, horizon], P_tracking)
    problem = cp.Problem(cp.Minimize(cost), constraints)
    problem.solve(solver=solver, warm_start=True, verbose=False, eps_abs=1e-5, eps_rel=1e-5, max_iter=20000)
    if U.value is None:
        return None, problem.status, None
    return float(U.value[0, 0]), problem.status, X.value


def simulate_disturbance_closed_loop(x0, horizon, sim_steps, d, disturbance_aware=False):
    x = np.array(x0, dtype=float).copy()
    xs = [x.copy()]
    us = []
    statuses = []
    for _ in range(sim_steps):
        if disturbance_aware:
            u_cmd, status, _ = solve_mpc_step_disturbance(x, d, horizon)
        else:
            u_cmd, status, _ = solve_mpc_step_nominal(x, horizon)
        statuses.append(status)
        if u_cmd is None:
            break
        x = A @ x + B[:, 0] * u_cmd + Bd[:, 0] * d
        xs.append(x.copy())
        us.append(u_cmd)
        if np.any(x < x_min - 1e-8) or np.any(x > x_max + 1e-8):
            break
    return {"x": np.array(xs), "u": np.array(us), "statuses": statuses}


def classify_disturbance_point(x0, horizon, sim_steps, d, disturbance_aware=False):
    sim = simulate_disturbance_closed_loop(x0, horizon, sim_steps, d, disturbance_aware)
    if len(sim["u"]) == 0:
        return 0, sim
    if len(sim["u"]) < sim_steps:
        return 1, sim
    final_error = np.linalg.norm(sim["x"][-1])
    return (2 if final_error <= convergence_tolerance else 1), sim


def evaluate_disturbance_map(horizon, sim_steps, d, disturbance_aware=False, grid_n=map_grid_n, progress=None):
    xs = np.linspace(-4.0, 4.0, grid_n)
    ys = np.linspace(-4.0, 4.0, grid_n)
    X1, X2 = np.meshgrid(xs, ys)
    class_map = np.zeros_like(X1, dtype=int)
    total = grid_n * grid_n
    if progress is not None:
        progress.max = total
        progress.value = 0
    count = 0
    for i in range(grid_n):
        for j in range(grid_n):
            x0 = np.array([xs[i], ys[j]], dtype=float)
            cls, _ = classify_disturbance_point(x0, horizon, sim_steps, d, disturbance_aware)
            class_map[j, i] = cls
            count += 1
            if progress is not None:
                progress.value = count
    return X1, X2, class_map


def generate_disturbance_trajectories(horizon, sim_steps, d, disturbance_aware=False, sample_n=9, progress=None):
    xs = np.linspace(-4.0, 4.0, sample_n)
    ys = np.linspace(-4.0, 4.0, sample_n)
    trajectories = []
    total = sample_n * sample_n
    count = 0
    if progress is not None:
        progress.max = total
        progress.value = 0
    for x1 in xs:
        for x2 in ys:
            x0 = np.array([x1, x2], dtype=float)
            cls, sim = classify_disturbance_point(x0, horizon, sim_steps, d, disturbance_aware)
            trajectories.append({"class": cls, "sim": sim, "x0": x0})
            count += 1
            if progress is not None:
                progress.value = count
    return trajectories


def plot_disturbance_map(ax, X1, X2, class_map, trajectories, title):
    region_colors = ["#bdbdbd", "#f99d44", "#2e7d32"]
    ax.contourf(X1, X2, class_map, levels=[-0.5, 0.5, 1.5, 2.5], colors=region_colors, alpha=0.9)
    feasible_mask = (class_map >= 1).astype(float)
    ax.contour(X1, X2, feasible_mask, levels=[0.5], colors="black", linewidths=1.8)
    for traj in trajectories:
        states = traj["sim"]["x"]
        if len(states) < 2:
            continue
        traj_color = "#1f1f1f" if traj["class"] == 2 else "#6a1b9a"
        ax.plot(states[:, 0], states[:, 1], color=traj_color, lw=0.9, alpha=0.35)
    ax.set_title(title)
    ax.set_xlabel("$x_1$")
    ax.set_xlim(-4.0, 4.0)
    ax.set_ylim(-4.0, 4.0)
    ax.grid(True, alpha=0.35)


def render_disturbance_results():
    nominal_map_progress = widgets.IntProgress(value=0, min=0, max=map_grid_n * map_grid_n, description="Nominal map:")
    aware_map_progress = widgets.IntProgress(value=0, min=0, max=map_grid_n * map_grid_n, description="Aware map:")
    nominal_traj_progress = widgets.IntProgress(value=0, min=0, max=81, description="Nominal traj:")
    aware_traj_progress = widgets.IntProgress(value=0, min=0, max=81, description="Aware traj:")
    progress_box = widgets.VBox([
        widgets.Label("Computing disturbance feasibility maps and sample trajectories..."),
        nominal_map_progress,
        aware_map_progress,
        nominal_traj_progress,
        aware_traj_progress,
    ])
    display(progress_box)
    X1_dist, X2_dist, nominal_disturbance_map = evaluate_disturbance_map(
        horizon_dist,
        sim_steps_dist,
        d_true,
        False,
        map_grid_n,
        progress=nominal_map_progress,
    )
    _, _, disturbance_aware_map = evaluate_disturbance_map(
        horizon_dist,
        sim_steps_dist,
        d_true,
        True,
        map_grid_n,
        progress=aware_map_progress,
    )
    nominal_disturbance_traj = generate_disturbance_trajectories(
        horizon_dist,
        sim_steps_dist,
        d_true,
        False,
        sample_n=9,
        progress=nominal_traj_progress,
    )
    disturbance_aware_traj = generate_disturbance_trajectories(
        horizon_dist,
        sim_steps_dist,
        d_true,
        True,
        sample_n=9,
        progress=aware_traj_progress,
    )
    progress_box.children = [widgets.Label("Rendering disturbance feasibility maps..."), nominal_map_progress, aware_map_progress, nominal_traj_progress, aware_traj_progress]
    fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)
    plot_disturbance_map(
        axes[0],
        X1_dist,
        X2_dist,
        nominal_disturbance_map,
        nominal_disturbance_traj,
        "Nominal MPC with disturbance",
    )
    plot_disturbance_map(
        axes[1],
        X1_dist,
        X2_dist,
        disturbance_aware_map,
        disturbance_aware_traj,
        "Disturbance-aware MPC",
    )
    axes[0].set_ylabel("$x_2$")
    legend_handles = [
        plt.Line2D([0], [0], color="#bdbdbd", lw=8, label="Infeasible at k = 0"),
        plt.Line2D([0], [0], color="#f99d44", lw=8, label="Feasible but not converged"),
        plt.Line2D([0], [0], color="#2e7d32", lw=8, label="Converges to zero"),
        plt.Line2D([0], [0], color="black", lw=1.8, label="Initial feasible boundary"),
        plt.Line2D([0], [0], color="#1f1f1f", lw=1.4, label="Convergent sample trajectories"),
        plt.Line2D([0], [0], color="#6a1b9a", lw=1.4, label="Non-convergent sample trajectories"),
    ]
    fig.legend(handles=legend_handles, loc="lower center", ncol=3, framealpha=0.95)
    fig.tight_layout(rect=[0, 0.12, 1, 1])
    plt.show()


disturbance_out = widgets.Output()

display(disturbance_out)

def update_disturbance_view():
    disturbance_out.clear_output(wait=True)
    with disturbance_out:
        render_disturbance_results()

update_disturbance_view()
